Bakoulas Epameinondas - AEM: 10683

# Assignment 2

## Optimal Warehouse Location

First, we will import all the necessary data that we are given. We will also use the libraries `gurobipy` to solve the optimization problem and `pandas` to display our results.

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

num_warehouses = 12
num_sales_centers = 12

warehouses = [f"W{i+1}" for i in range(num_warehouses)]
sales_centers = [f"SC{i+1}" for i in range(num_sales_centers)]

inf = float('inf')
total_transport_costs_data = [
    [100,  80,  50,  50,  60, 100, 120,  90,  60,  70,  65, 110],
    [120,  90,  60,  70,  65, 110, 140, 110,  80,  80,  75, 130],
    [140, 110,  80,  80,  75, 130, 160, 125, 100, 100,  80, 150],
    [160, 125, 100, 100,  80, 150, 190, 150, 130, inf, inf, inf],
    [190, 150, 130, inf, inf, inf, 180, 150,  50,  50,  60, 100],
    [200, 180, 150, inf, inf, inf, 100, 120,  90,  60,  75, 110],
    [120,  90,  60,  70,  65, 110, 140, 110,  80,  80,  75, 130],
    [120,  90,  60,  70,  65, 110, 140, 110,  80,  80,  75, 130],
    [140, 110,  80,  80,  75, 130, 160, 125, 100, 100,  80, 150],
    [160, 125, 100, 100,  80, 150, 190, 150, 130, inf, inf, inf],
    [190, 150, 130, inf, inf, inf, 200, 180, 150, inf, inf, inf],
    [200, 180, 150, inf, inf, inf, 100,  80,  50,  50,  60, 100]
]

fixed_costs_data = [3500, 9000, 10000, 4000, 3000, 9000, 9000, 3000, 4000, 10000, 9000, 3500]
capacities_data = [300, 250, 100, 180, 275, 300, 200, 220, 270, 250, 230, 180]
demands_data = [120, 80, 75, 100, 110, 100, 90, 60, 30, 150, 95, 120]

# Convert to dictionaries for easier access
fixed_costs = {warehouses[i]: fixed_costs_data[i] for i in range(num_warehouses)}
capacities = {warehouses[i]: capacities_data[i] for i in range(num_warehouses)}
demands = {sales_centers[j]: demands_data[j] for j in range(num_sales_centers)}

### Unit transportation costs

Now we're going to calculate the **unit transportation costs** (in thousands of Euros per ton), $u_{ij}$ where $i$ is the warehouse number and $j$ is the sales center.

These are calculated with the following formula:

$$
u_{ij} = \frac{tc_{ij}}{d_j}
$$

where $tc_{ij}$ is the transportation cost from warehouse $i$ to sales center $j$ (that satisfied the full demand of the sales center) and $d_j$ is the demand of sales center $j$.

In [2]:
unit_transport_costs = {}
for i in range(num_warehouses):
    for j in range(num_sales_centers):
        w = warehouses[i]
        sc = sales_centers[j]
        tc_ij = total_transport_costs_data[i][j]
        d_j = demands[sc]
        if tc_ij != inf:
            unit_transport_costs[(w, sc)] = tc_ij / d_j
        else:
            unit_transport_costs[(w, sc)] = inf

### Decision Variables

We're now going to create the model, and add the variables.

There are **two types of variables, X and Y**.

$X_{ij}$ represents the **quantity shipped** from warehouse $i$ to sales center $j$ (in tons). It's a **continuous variable**. If the cost associated with this route is $inf$, then we don't create this specific $X_{ij}$ variable. We also define our first constraint here, $X_{ij} > 0$ (using lb=0).

$Y_{i}$ represents the **binary variable** that indicates **whether warehouse $i$ is open or not**. If $Y_{i}$ is 1, then warehouse $i$ is open.

In [3]:
model = gp.Model("WarehouseLocation")

ship_routes = [(w, sc) 
               for w in warehouses 
               for sc in sales_centers 
               if unit_transport_costs[(w, sc)] != inf]
X = model.addVars(ship_routes, vtype=GRB.CONTINUOUS, name="Ship", lb=0) # lb=0 for non-negative

Y = model.addVars(warehouses, vtype=GRB.BINARY, name="Open")


Set parameter Username
Set parameter LicenseID to value 2668025
Academic license - for non-commercial use only - expires 2026-05-19


### Objective Function

The objective function is to **minimize the total cost** of the system, which is the sum of the transportation costs and the fixed costs of opening the warehouses.

So we have a linear function of the form:
$$
\text{minimize } \sum_{i=1}^{n} \sum_{j=1}^{m} u_{ij} X_{ij} + \sum_{i=1}^{n} fc_i Y_i
$$
where $fc_i$ is the fixed cost of opening warehouse $i$.

In [4]:
obj_fixed_costs = gp.quicksum(fixed_costs[w] * Y[w] 
                              for w in warehouses)

obj_transport_costs = gp.quicksum(unit_transport_costs[(w, sc)] * X[(w, sc)] 
                                  for w, sc in ship_routes)

model.setObjective(obj_fixed_costs + obj_transport_costs, GRB.MINIMIZE)

### Constraints

Our first constraint is that the sum of shipments (in tons) to a specific sales center must match the demand. This is expressed as:

$$
\sum_{i=1}^{n} X_{ij} = d_j \quad \forall j
$$
where $d_j$ is the demand of sales center $j$.

In [5]:
for sc in sales_centers:
    model.addConstr(
        gp.quicksum(X[(w, sc)] 
                    for w in warehouses 
                    if (w, sc) in ship_routes) == demands[sc]
    )

The 2nd constraint is that the total shipments (in tons) from a specific warehouse must not exceed the capacity of that warehouse. This is expressed as:

$$
\sum_{j=1}^{m} X_{ij} \leq c_i Y_i \quad \forall i
$$
where $c_i$ is the capacity of warehouse $i$.

If the warehouse $i$ is closed, then $Y_i = 0$, and the constraint becomes $\sum_{j=1}^{m} X_{ij} \leq 0$. Since $X_{ij}$ is a non-negative variable, this means that no shipments can be made from warehouse $i$ ($X_{ij} = 0$).

In [6]:
for w in warehouses:
    model.addConstr(
        gp.quicksum(X[(w, sc)] 
                    for sc in sales_centers 
                    if (w, sc) in ship_routes) <= capacities[w] * Y[w]
    )

Finally, we solve the model.

In [ ]:
model.optimize()

## Results

We will now display the minimum cost (in kEuros), the warehouses that we should open, and the shipments from each open warehouse to each sales center (in tons).

In [8]:
if model.status == GRB.OPTIMAL:
    print(f"Total Minimum Cost: {model.ObjVal:.2f}k Euros")

    print("\nWarehouses to Open:")
    for w in warehouses:
        if Y[w].X == 1:
            print(f"- {w}")

    print("\nShipment Plan (tons):")
    shipment_data = []
    for w_idx, w in enumerate(warehouses):
        if Y[w].X == 1:
            row = {'Warehouse': w}
            total_shipped_from_w = 0
            for sc_idx, sc in enumerate(sales_centers):
                quantity = 0
                if (w,sc) in X:
                    quantity = X[(w,sc)].X
                row[sc] = f"{quantity:.2f}"
                total_shipped_from_w += quantity
            row['Total Shipped'] = f"{total_shipped_from_w:.2f}"
            shipment_data.append(row)

    shipment_df = pd.DataFrame(shipment_data)
    cols = ['Warehouse'] + sales_centers + ['Total Shipped']
    shipment_df = shipment_df[cols]
    display(shipment_df)

Total Minimum Cost: 17929.23k Euros

Warehouses to Open:
- W1
- W5
- W8
- W9
- W12

Shipment Plan (tons):


,Warehouse,SC1,SC2,SC3,SC4,SC5,SC6,SC7,SC8,SC9,SC10,SC11,SC12,Total Shipped
0,W1,120.00,5.00,75.00,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,300.00
1,W5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,150.00,5.00,120.00,275.00
2,W8,0.00,75.00,0.00,0.00,45.00,100.00,0.00,0.00,0.00,0.00,0.00,0.00,220.00
3,W9,0.00,0.00,0.00,0.00,65.00,0.00,0.00,0.00,0.00,0.00,90.00,0.00,155.00
4,W12,0.00,0.00,0.00,0.00,0.00,0.00,90.00,60.00,30.00,0.00,0.00,0.00,180.00
